In [ ]:
"""
Imports
"""

import laspy
import open3d as o3d
import numpy as np
import pyntcloud
from pyntcloud import PyntCloud
import pandas as pd

In [ ]:
"""
Variables and filepaths 
"""

final_project_segment_file = "final_project_segment.laz"


In [ ]:
def read_las_points(filepath):
    las = laspy.read(filepath)
    points = np.vstack((las.x, las.y, las.z)).T
    return las, points

In [ ]:
las, points = read_las_points(final_project_segment_file)
print("Number of points: ", len(las.points))

In [ ]:
"""
2. Outlier removal (Statistical)
"""
# Konverter til Open3D point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# Fjern uteliggere: 10 naboer, fjern punkter som ligger > 5 std-avvik fra snittet
cl, ind = pcd.remove_statistical_outlier(nb_neighbors=50, std_ratio=5.0)

# Behold kun punktene som ikke ble fjernet
pcd_filtered = pcd.select_by_index(ind)
print(f"Removed {len(points) - len(ind):,} outlier points")
print(f"Remaining points after filtering: {len(ind):,}")

# Konverter tilbake til numpy-array
points_filtered = np.asarray(pcd_filtered.points)

"""
3. Create PyntCloud object from filtered points
"""
# Lag dataframe for PyntCloud
df = pd.DataFrame(points_filtered, columns=["x", "y", "z"])
cloud = PyntCloud(df)

"""
4. Voxelization
"""
voxel_size = 2  # meter
voxelgrid_id = cloud.add_structure("voxelgrid",
                                   size_x=voxel_size,
                                   size_y=voxel_size,
                                   size_z=voxel_size)
voxelgrid = cloud.structures[voxelgrid_id]

num_voxels = len(voxelgrid.voxel_n)
print(f"Total voxels created: {num_voxels:,}")

"""
5. Filter out voxels with fewer than 10 points
"""
valid_voxel_mask = voxelgrid.voxel_n > 10
valid_voxel_indices = np.where(valid_voxel_mask)[0]
num_valid_voxels = len(valid_voxel_indices)
print(f"Voxels with >=10 points: {num_valid_voxels:,}")

"""
6. Create a filtered point cloud with only points from valid voxels (manual mapping)
"""
# Finn min- og makskoordinater for punktene
min_coords = np.floor(points_filtered.min(axis=0))
max_coords = np.ceil(points_filtered.max(axis=0))

# Beregn hvor mange voxler vi har i hver retning (avrundet opp)
dims = np.ceil((max_coords - min_coords) / voxel_size).astype(int)

# Beregn hvilken voxel hvert punkt tilhører
coords = np.floor((points_filtered - min_coords) / voxel_size).astype(int)

# Sørg for at alle punkter havner innenfor gridet
coords = np.clip(coords, 0, dims - 1)

# Gi hvert punkt et unikt voxel-ID
voxel_ids = np.ravel_multi_index((coords[:,0], coords[:,1], coords[:,2]), dims=dims)

# Tell hvor mange punkter hver voxel inneholder
unique_ids, counts = np.unique(voxel_ids, return_counts=True)
voxel_count_dict = dict(zip(unique_ids, counts))

# Filtrer bort punkter som ligger i voxler med færre enn 10 punkter
mask_valid = np.array([voxel_count_dict.get(i, 0) >= 10 for i in voxel_ids])
filtered_points_final = points_filtered[mask_valid]

print(f"Points after voxel filtering: {filtered_points_final.shape[0]:,}")
print(f"Kept {(filtered_points_final.shape[0] / len(points_filtered)) * 100:.2f}% of points after voxel filtering")
 
"""
7. Save or visualize (optional)
"""
# Lag nytt Open3D-punktcloud for visning
pcd_voxel_filtered = o3d.geometry.PointCloud()
pcd_voxel_filtered.points = o3d.utility.Vector3dVector(filtered_points_final)

# Visualiser
o3d.visualization.draw_geometries([pcd_voxel_filtered])
unique_voxel_ids = np.unique(voxel_ids)

print("Unique occupied voxels:", len(unique_voxel_ids))
print(f"Removed {len(points) - len(ind)} outliers ({(len(points) - len(ind))/len(points)*100:.3f}%)")



In [ ]:
from sklearn.decomposition import PCA

# 1️⃣ Samle punkter per voxel
voxel_ids_filtered = voxel_ids[mask_valid]
unique_voxel_ids = np.unique(voxel_ids_filtered)

voxel_features = []

for voxel_id in unique_voxel_ids:
    voxel_points = filtered_points_final[voxel_ids_filtered == voxel_id]
    
    # Hopper over små voxler (sikkerhetsmargin)
    if len(voxel_points) < 10:
        continue

    # 2️⃣ PCA på punktene
    pca = PCA(n_components=3)
    pca.fit(voxel_points)
    eigenvalues = np.sort(pca.explained_variance_)[::-1]  # sortert: λ1 ≥ λ2 ≥ λ3

    λ1, λ2, λ3 = eigenvalues
    if λ1 == 0:
        continue

    # 3️⃣ Beregn features
    linearity = (λ1 - λ2) / λ1
    planarity = (λ2 - λ3) / λ1
    scattering = λ3 / λ1
    omnivariance = (λ1 * λ2 * λ3) ** (1/3)
    sum_ev = λ1 + λ2 + λ3
    anisotropy = (λ1 - λ3) / λ1
    eigentropy = -(λ1*np.log(λ1) + λ2*np.log(λ2) + λ3*np.log(λ3))
    change_curvature = λ3 / (λ1 + λ2 + λ3)

    # 4️⃣ Ikke-PCA features
    z_range = voxel_points[:,2].max() - voxel_points[:,2].min()

    # Standard deviation from local plane = std i z-retning etter PCA
    plane_std = np.std(pca.transform(voxel_points)[:, 2])

    # Lagre alt i én dictionary
    voxel_features.append({
        "voxel_id": voxel_id,
        "n_points": len(voxel_points),
        "linearity": linearity,
        "planarity": planarity,
        "scattering": scattering,
        "omnivariance": omnivariance,
        "sum_ev": sum_ev,
        "anisotropy": anisotropy,
        "eigentropy": eigentropy,
        "change_curvature": change_curvature,
        "z_range": z_range,
        "plane_std": plane_std
    })

print(f"Calculated features for {len(voxel_features)} voxels")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_features = pd.DataFrame(voxel_features)

# Lag histogram for hver feature
df_features.hist(bins=30, figsize=(12,10))
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

df_features["scattering_log2"] = np.log2(df_features["scattering"] + 1e-6)
df_features["plane_std_log2"] = np.log2(df_features["plane_std"] + 1e-6)

df_features[["scattering", "scattering_log2"]].hist(bins=30)
df_features[["plane_std", "plane_std_log2"]].hist(bins=30)
plt.show()


In [ ]:
"""
Save voxel centers as .las for visualization in CloudCompare
"""

import laspy

# Beregn senterkoordinat for hver voxel (for visualisering)
voxel_centers = []
for voxel_id in unique_voxel_ids:
    voxel_points = filtered_points_final[voxel_ids_filtered == voxel_id]
    center = voxel_points.mean(axis=0)
    voxel_centers.append(center)

voxel_centers = np.array(voxel_centers)

# Lagre som .las slik at du kan åpne i CloudCompare
header = laspy.LasHeader(point_format=3, version="1.2")
las = laspy.LasData(header)
las.x, las.y, las.z = voxel_centers[:,0], voxel_centers[:,1], voxel_centers[:,2]
las.write("voxel_centers.las")

print(len(voxel_centers), "voxel centers saved to voxel_centers.las")

print("✅ Saved voxel centers to voxel_centers.las")

In [ ]:
def read_las_coords(filepath):
    las = laspy.read(filepath)
    return np.vstack((las.x, las.y, las.z)).T

buildings = read_las_coords("voxel_centers_building.las")
trees = read_las_coords("voxel_centers_trees.las")
terrain = read_las_coords("voxel_centers_terrain.las")

def find_nearest_voxel_ids(coords, voxel_centers, voxel_ids):
    from scipy.spatial import cKDTree
    tree = cKDTree(voxel_centers)
    _, idx = tree.query(coords, k=1)
    return np.array(voxel_ids)[idx]

building_ids = find_nearest_voxel_ids(buildings, voxel_centers, unique_voxel_ids)
tree_ids = find_nearest_voxel_ids(trees, voxel_centers, unique_voxel_ids)
terrain_ids = find_nearest_voxel_ids(terrain, voxel_centers, unique_voxel_ids)

df_features["label"] = "unlabeled"
df_features.loc[df_features["voxel_id"].isin(building_ids), "label"] = "building"
df_features.loc[df_features["voxel_id"].isin(tree_ids), "label"] = "tree"
df_features.loc[df_features["voxel_id"].isin(terrain_ids), "label"] = "terrain"

train_df = df_features[df_features["label"] != "unlabeled"]
print(train_df["label"].value_counts())

train_df.to_csv("voxel_features_labeled.csv", index=False)
print("✅ Labeled training data saved to voxel_features_labeled.csv")


In [ ]:
"""
Feature Analysis – PCA-baserte 3D-features per voxel
Author: Åsne Haavengen
"""

# -------------------- Imports --------------------
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import open3d as o3d

# -------------------- 1. Forutsetning --------------------
# Denne koden forutsetter at du allerede har gjort:
#  - Outlier removal
#  - Voxelization
# og at du har følgende variabler klare:
#     points_filtered        (etter uteligger-fjerning)
#     voxel_ids              (tilordnet hvert punkt)
#     mask_valid             (True for punkter i voxler med ≥10 punkter)

# -------------------- 2. Filtrér punkter og voxel-IDer --------------------
filtered_points_final = points_filtered[mask_valid]
voxel_ids_filtered = voxel_ids[mask_valid]

print(f"Points after voxel filtering: {filtered_points_final.shape[0]:,}")
print(f"Unique voxels after filtering: {len(np.unique(voxel_ids_filtered)):,}")

# 💾 LAGRE til bruk i neste steg (klassifisering)
np.save("filtered_points_final.npy", filtered_points_final)
np.save("voxel_ids_filtered.npy", voxel_ids_filtered)
print("✅ Saved filtered_points_final.npy and voxel_ids_filtered.npy")

# -------------------- 3. Beregn PCA-features per voxel --------------------
voxel_features = []
unique_voxel_ids = np.unique(voxel_ids_filtered)

for voxel_id in unique_voxel_ids:
    voxel_points = filtered_points_final[voxel_ids_filtered == voxel_id]
    if len(voxel_points) < 10:
        continue

    # PCA
    pca = PCA(n_components=3)
    pca.fit(voxel_points)
    eigenvalues = np.sort(pca.explained_variance_)[::-1]
    λ1, λ2, λ3 = eigenvalues
    if λ1 == 0:
        continue

    # PCA-baserte features
    linearity = (λ1 - λ2) / λ1
    planarity = (λ2 - λ3) / λ1
    scattering = λ3 / λ1
    omnivariance = (λ1 * λ2 * λ3) ** (1/3)
    sum_ev = λ1 + λ2 + λ3
    anisotropy = (λ1 - λ3) / λ1
    eigentropy = -(λ1*np.log(λ1) + λ2*np.log(λ2) + λ3*np.log(λ3))
    change_curvature = λ3 / (λ1 + λ2 + λ3)

    # Ikke-PCA-baserte
    z_range = voxel_points[:, 2].max() - voxel_points[:, 2].min()
    plane_std = np.std(pca.transform(voxel_points)[:, 2])

    voxel_features.append({
        "voxel_id": voxel_id,
        "n_points": len(voxel_points),
        "linearity": linearity,
        "planarity": planarity,
        "scattering": scattering,
        "omnivariance": omnivariance,
        "sum_ev": sum_ev,
        "anisotropy": anisotropy,
        "eigentropy": eigentropy,
        "change_curvature": change_curvature,
        "z_range": z_range,
        "plane_std": plane_std
    })

print(f"✅ Calculated features for {len(voxel_features)} voxels")

# -------------------- 4. Lag DataFrame --------------------
df_features = pd.DataFrame(voxel_features)
df_features.to_csv("voxel_features_all.csv", index=False)
print("💾 Saved voxel_features_all.csv")

# -------------------- 5. Beregn voxel-sentre i samme rekkefølge --------------------
voxel_centers = []
for voxel_id in df_features["voxel_id"].values:
    pts = filtered_points_final[voxel_ids_filtered == voxel_id]
    voxel_centers.append(pts.mean(axis=0))

voxel_centers = np.array(voxel_centers)
np.save("voxel_centers.npy", voxel_centers)
print(f"💾 Saved voxel_centers.npy for {len(voxel_centers)} voxels")

# -------------------- 6. Visualiser (valgfritt) --------------------
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(voxel_centers)
o3d.visualization.draw_geometries([pcd])


In [ ]:
"""
CatBoost classification of 3D voxels
Author: Åsne Haavengen
"""

# -------------------- Imports --------------------
import numpy as np
import pandas as pd
import open3d as o3d
import laspy
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# -------------------- 1. Reconstruct voxel centers to match df --------------------
# Denne delen sikrer at voxel_centers.npy passer perfekt med feature-radene
# Kjør denne delen i samme prosjektmappe der du har de nødvendige filene:
#   - "filtered_points_final.npy"   (punktene etter voxel-filtrering)
#   - "voxel_ids_filtered.npy"      (voxel-ID for hvert punkt)
#   - "voxel_features_labeled.csv"  (featuretabell)

# Hvis du allerede har lastet inn punkter og voxel_ids, kan du hoppe til neste steg
points_filtered = np.load("filtered_points_final.npy")       # (N, 3)
voxel_ids_filtered = np.load("voxel_ids_filtered.npy")       # (N,)

# Les featurefil
df = pd.read_csv("voxel_features_labeled.csv")
print(f"Loaded {len(df)} voxel features")

# Sørg for at df inneholder kolonnen "voxel_id"
assert "voxel_id" in df.columns, "Feature-filen må inneholde 'voxel_id'-kolonnen"

# Beregn senter for hver voxel i samme rekkefølge som i df
voxel_centers = []
for voxel_id in df["voxel_id"].values:
    pts = points_filtered[voxel_ids_filtered == voxel_id]
    if len(pts) == 0:
        voxel_centers.append([np.nan, np.nan, np.nan])
    else:
        voxel_centers.append(pts.mean(axis=0))

voxel_centers = np.array(voxel_centers)
np.save("voxel_centers.npy", voxel_centers)
print(f"Saved {len(voxel_centers)} voxel centers (matching df order)")

# Fjern eventuelle NaN-rader
mask_valid = ~np.isnan(voxel_centers).any(axis=1)
df = df[mask_valid].reset_index(drop=True)
voxel_centers = voxel_centers[mask_valid]

# -------------------- 2. Feature engineering --------------------
# Log2-transformasjon på utvalgte features
df["scattering_log2"] = np.log2(df["scattering"] + 1e-6)
df["plane_std_log2"] = np.log2(df["plane_std"] + 1e-6)

# Featureliste
feature_cols = [
    "linearity", "planarity", "scattering_log2", "omnivariance", "sum_ev",
    "anisotropy", "eigentropy", "change_curvature", "z_range", "plane_std_log2"
]

# -------------------- 3. Split training and test data --------------------
train_df = df[df["label"] != "unlabeled"]
X = train_df[feature_cols]
y = train_df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining samples: {len(X_train)}, Test samples: {len(X_test)}")

# -------------------- 4. Train CatBoost model --------------------
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    verbose=100
)

model.fit(X_train, y_train)

# -------------------- 5. Evaluate model --------------------
y_pred = model.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_pred))

# -------------------- 6. Classify all voxels --------------------
X_all = df[feature_cols]
preds = model.predict(X_all)
df["pred_label"] = preds.flatten()

print("\nPredicted label counts:\n", df["pred_label"].value_counts())

# -------------------- 7. Visualize in 3D --------------------
color_map = {
    "building": [1.0, 0.0, 0.0],  # rød
    "tree": [0.0, 1.0, 0.0],      # grønn
    "terrain": [0.5, 0.25, 0.0]   # brun
}

colors = np.array([color_map.get(lbl, [0.6, 0.6, 0.6]) for lbl in df["pred_label"].values])

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(voxel_centers)
pcd.colors = o3d.utility.Vector3dVector(colors)

print("\nVisualizing classified voxels...")
o3d.visualization.draw_geometries([pcd])

# -------------------- 8. Save classified voxels --------------------
las = laspy.create(point_format=3)
las.x, las.y, las.z = voxel_centers[:,0], voxel_centers[:,1], voxel_centers[:,2]

rgb = (colors * 255).astype(np.uint16)
las.red, las.green, las.blue = rgb[:,0], rgb[:,1], rgb[:,2]

las.write("classified_voxels.las")
print("✅ Saved classified voxels to 'classified_voxels.las'")

# -------------------- 9. Save full feature dataset with predictions --------------------
df.to_csv("voxel_features_classified.csv", index=False)
print("💾 Saved classified voxel features to 'voxel_features_classified.csv'")


In [ ]:
# -------------------- 10. Apply model to all points --------------------
print("\nApplying voxel predictions to all points...")

# Last punkter og voxel-tilordninger
points_filtered = np.load("filtered_points_final.npy")
voxel_ids_filtered = np.load("voxel_ids_filtered.npy")

# Lag oppslagstabell: voxel_id -> pred_label
voxel_label_map = dict(zip(df["voxel_id"].values, df["pred_label"].values))

# Lag array med label for hvert punkt
point_labels = np.array([voxel_label_map.get(v_id, "unclassified") for v_id in voxel_ids_filtered])

# Lag farger per punkt
color_map = {
    "building": [1.0, 0.0, 0.0],   # rød
    "tree": [0.0, 1.0, 0.0],       # grønn
    "terrain": [0.5, 0.25, 0.0],   # brun
    "unclassified": [0.7, 0.7, 0.7]
}

colors_points = np.array([color_map.get(lbl, [0.6, 0.6, 0.6]) for lbl in point_labels])

# -------------------- 11. Visualize all classified points --------------------
pcd_points = o3d.geometry.PointCloud()
pcd_points.points = o3d.utility.Vector3dVector(points_filtered)
pcd_points.colors = o3d.utility.Vector3dVector(colors_points)

print("🌈 Visualizing all classified points...")
o3d.visualization.draw_geometries([pcd_points])

# -------------------- 12. Save full classified point cloud --------------------
las = laspy.create(point_format=3)
las.x, las.y, las.z = points_filtered[:, 0], points_filtered[:, 1], points_filtered[:, 2]

rgb_points = (colors_points * 255).astype(np.uint16)
las.red, las.green, las.blue = rgb_points[:, 0], rgb_points[:, 1], rgb_points[:, 2]

las.write("classified_points.las")
print("✅ Saved all classified points to 'classified_points.las'")


In [ ]:
"""
Final 3D voxel classification with CatBoost
Author: Åsne Haavengen
"""

# -------------------- Imports --------------------
import numpy as np
import pandas as pd
import open3d as o3d
import laspy
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.spatial import cKDTree


# -------------------- 1. Load all data --------------------
points_filtered = np.load("filtered_points_final.npy")       # (N_points, 3)
voxel_ids_filtered = np.load("voxel_ids_filtered.npy")       # (N_points,)
df = pd.read_csv("voxel_features_labeled.csv")

assert "voxel_id" in df.columns, "Feature-filen må inneholde 'voxel_id'"

print(f"Loaded {len(df)} voxel features, {len(points_filtered)} points")


# -------------------- 2. Recompute voxel centers --------------------
voxel_centers = []
for voxel_id in df["voxel_id"].values:
    pts = points_filtered[voxel_ids_filtered == voxel_id]
    if len(pts) == 0:
        voxel_centers.append([np.nan, np.nan, np.nan])
    else:
        voxel_centers.append(pts.mean(axis=0))

voxel_centers = np.array(voxel_centers)
mask_valid = ~np.isnan(voxel_centers).any(axis=1)
df = df[mask_valid].reset_index(drop=True)
voxel_centers = voxel_centers[mask_valid]
np.save("voxel_centers.npy", voxel_centers)

print(f"Computed {len(voxel_centers)} voxel centers matching features")


# -------------------- 3. Feature engineering --------------------
df["scattering_log2"] = np.log2(df["scattering"] + 1e-6)
df["plane_std_log2"]  = np.log2(df["plane_std"] + 1e-6)

feature_cols = [
    "linearity", "planarity", "scattering_log2", "omnivariance", "sum_ev",
    "anisotropy", "eigentropy", "change_curvature", "z_range", "plane_std_log2"
]


# -------------------- 4. Train CatBoost model --------------------
train_df = df[df["label"] != "unlabeled"]
X = train_df[feature_cols]
y = train_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining samples: {len(X_train)}, Test samples: {len(X_test)}")

model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    verbose=100
)

model.fit(X_train, y_train)

# --- Evaluate ---
y_pred = model.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_pred))


# -------------------- 5. Predict for all voxels --------------------
X_all = df[feature_cols]
df["pred_label"] = model.predict(X_all).flatten()
print("\nPredicted label counts:\n", df["pred_label"].value_counts())


# -------------------- 6. Visualize voxel-level result --------------------
color_map = {
    "building": [1.0, 0.0, 0.0],
    "tree": [0.0, 1.0, 0.0],
    "terrain": [0.5, 0.25, 0.0],
    "unclassified": [0.7, 0.7, 0.7]
}
colors_voxel = np.array([color_map.get(lbl, [0.6, 0.6, 0.6]) for lbl in df["pred_label"].values])

pcd_voxel = o3d.geometry.PointCloud()
pcd_voxel.points = o3d.utility.Vector3dVector(voxel_centers)
pcd_voxel.colors = o3d.utility.Vector3dVector(colors_voxel)
print("\nVisualizing classified voxels...")
o3d.visualization.draw_geometries([pcd_voxel])


# -------------------- 7. Assign predicted labels to all points --------------------
print("\nAssigning predicted labels to all points...")

# KDTree-metode (robust selv om voxel_id ikke stemmer helt)
voxel_tree = cKDTree(voxel_centers)
distances, indices = voxel_tree.query(points_filtered, k=1)
point_labels = df["pred_label"].values[indices]

# Farger
colors_points = np.array([color_map.get(lbl, [0.6, 0.6, 0.6]) for lbl in point_labels])

# Visualiser
pcd_points = o3d.geometry.PointCloud()
pcd_points.points = o3d.utility.Vector3dVector(points_filtered)
pcd_points.colors = o3d.utility.Vector3dVector(colors_points)
print("🌈 Visualizing all classified points...")
o3d.visualization.draw_geometries([pcd_points])

# Oppsummer
unique, counts = np.unique(point_labels, return_counts=True)
print("\nPoint-level classification summary:")
for lbl, cnt in zip(unique, counts):
    print(f"{lbl:10s}: {cnt:,}")


# -------------------- 8. Save results --------------------
# --- Voxels (.las) ---
las_vox = laspy.create(point_format=3)
las_vox.x, las_vox.y, las_vox.z = voxel_centers[:,0], voxel_centers[:,1], voxel_centers[:,2]
rgb_vox = (colors_voxel * 255).astype(np.uint16)
las_vox.red, las_vox.green, las_vox.blue = rgb_vox[:,0], rgb_vox[:,1], rgb_vox[:,2]
las_vox.write("classified_voxels.las")
print("✅ Saved classified_voxels.las")

# --- All points (.las) ---
las_pts = laspy.create(point_format=3)
las_pts.x, las_pts.y, las_pts.z = points_filtered[:,0], points_filtered[:,1], points_filtered[:,2]
rgb_pts = (colors_points * 255).astype(np.uint16)
las_pts.red, las_pts.green, las_pts.blue = rgb_pts[:,0], rgb_pts[:,1], rgb_pts[:,2]
las_pts.write("classified_points.las")
print("✅ Saved classified_points.las")

# --- Feature CSV ---
df.to_csv("voxel_features_classified.csv", index=False)
print("💾 Saved voxel_features_classified.csv")


In [ ]:

# Save denoised point cloud with all original attributes
output_file = "denoised_subset_points_full.laz"

# Keep only the filtered points
las_denoised = laspy.create(point_format=las.header.point_format, file_version=las.header.version)

# Copy coordinates
las_denoised.x = las.x[ind]
las_denoised.y = las.y[ind]
las_denoised.z = las.z[ind]

# Copy all additional attributes if they exist
for dimension in las.point_format.dimensions:
    if dimension.name not in ["X", "Y", "Z"]:
        setattr(las_denoised, dimension.name, getattr(las, dimension.name)[ind])

# Copy metadata
las_denoised.header.offsets = las.header.offsets
las_denoised.header.scales = las.header.scales
las_denoised.header.system_identifier = "Open3D Denoised"

las_denoised.write(output_file)
print(f"✅ Saved full denoised point cloud with attributes → {output_file}")

In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.cm as cm

# Anta at du allerede har disse:
# voxel_points, voxel_ids, voxel_size, voxel_centers

print("Creating voxel mesh with scalar colors (based on random point height)...")

# Hent unike voxler
unique_voxels = np.unique(voxel_ids)

# Velg ett tilfeldig punkt fra hver voxel
random_indices = []
for vid in unique_voxels:
    idx = np.where(voxel_ids == vid)[0]
    if len(idx) > 0:
        random_indices.append(np.random.choice(idx))

random_indices = np.array(random_indices)
sample_points = voxel_points[random_indices]

# Bruk z-verdi som scalar (høyde)
scalar_values = sample_points[:, 2]
norm = (scalar_values - scalar_values.min()) / (scalar_values.max() - scalar_values.min())

# Lag farger basert på høydeskala ()
colormap = cm.get_cmap('jet')
colors = colormap(norm)[:, :3]  # RGBA → RGB

# Beregn voxel-centre på nytt for sikkerhet
voxel_centers = np.array([
    voxel_points[voxel_ids == vid].mean(axis=0)
    for vid in unique_voxels
])

# Lag kube-mal
base_cube = o3d.geometry.TriangleMesh.create_box(width=voxel_size, height=voxel_size, depth=voxel_size)
base_cube.compute_vertex_normals()
base_cube.translate([-voxel_size / 2, -voxel_size / 2, -voxel_size / 2])

# Kombiner kuber med farge
all_voxels = o3d.geometry.TriangleMesh()
for center, color in zip(voxel_centers, colors):
    cube = base_cube.translate(center, relative=False)
    cube.paint_uniform_color(color)
    all_voxels += cube

# Lagre resultatet
output_file = "voxel_mesh_colored.ply"
o3d.io.write_triangle_mesh(output_file, all_voxels)
print(f"✅ Saved colored voxel mesh → {output_file}")
